# LeetCode 963 — Minimum Area Rectangle II bằng Python

Link đề: https://leetcode.com/problems/minimum-area-rectangle-ii/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Math (toán), Geometry (hình học)

Skeleton LeetCode:

```python
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        ...
```

Ghi chú: cho các điểm nguyên trên mặt phẳng, tìm **diện tích nhỏ nhất** của một hình chữ nhật có 4 đỉnh trong tập điểm, các cạnh **không nhất thiết song song trục**; không có thì trả `0`. Đáp án chấp nhận sai số `10^-5`. Quan sát quan trọng: vì đỉnh là điểm nguyên, **diện tích luôn là số nguyên** (`|tích có hướng|` của hai vectơ cạnh), nên ta có thể tính **chính xác bằng số nguyên** và chỉ đổi sang `float` ở bước trả về; khi kiểm thử so sánh với **sai số** như đề (`abs(a - b) <= 1e-5`). Đặc trưng của hình chữ nhật: hai đường chéo **cắt nhau tại trung điểm chung** và **bằng nhau**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm hình chữ nhật (bất kỳ hướng) có diện tích nhỏ nhất với 4 đỉnh thuộc tập điểm.
2. Nắm các công cụ Python cần thiết: số nguyên tuyệt đối chính xác (tránh `float`), tích vô hướng và tích có hướng của vectơ, `collections.defaultdict`, `itertools.combinations`/`permutations`, và `set` các tuple điểm.
3. Hiểu nguyên lý: tứ giác là hình chữ nhật ⇔ hai đường chéo có **cùng trung điểm** và **cùng độ dài**; khóa nhóm là `(tổng tọa độ hai đầu, bình phương độ dài)` hoàn toàn bằng số nguyên.
4. Đi qua 4 cách: thử mọi tập 4 điểm theo định nghĩa góc vuông (brute), liệt kê bộ ba có góc vuông và tìm đỉnh thứ tư, nhóm các đường chéo theo `(tâm, độ dài)` (cách nộp chính), và nhóm các cạnh theo **vectơ** để tìm hình bình hành rồi kiểm tra góc vuông; đối chiếu và so sánh số thực **có sai số**.
5. Tránh các lỗi kinh điển: chia trung điểm bằng `//` (mất thông tin), chỉ nhóm theo tâm mà không kiểm tra độ dài (nhận hình bình hành), chỉ xét hình chữ nhật song song trục (bài 939), và trả `inf` khi không có hình chữ nhật.

## 2. Đề bài & ví dụ

Cho mảng các điểm trên mặt phẳng `X-Y`, `points[i] = [xi, yi]`.

Trả về **diện tích nhỏ nhất** của một hình chữ nhật tạo từ các điểm này, với các cạnh **không nhất thiết song song** với trục tọa độ. Nếu không có hình chữ nhật nào, trả về `0`. Đáp án sai khác không quá `10^-5` được chấp nhận.

**Ví dụ 1:** `points = [[1,2],[2,1],[1,0],[0,1]]` -> `2.00000` (hình vuông nghiêng).

**Ví dụ 2:** `points = [[0,1],[2,1],[1,1],[1,0],[2,0]]` -> `1.00000` (hình chữ nhật `[1,0],[1,1],[2,1],[2,0]`).

**Ví dụ 3:** `points = [[0,3],[1,2],[3,1],[1,3],[2,1]]` -> `0` (không có hình chữ nhật).

**Ràng buộc:**

- `1 <= points.length <= 50`
- `points[i].length == 2`
- `0 <= xi, yi <= 4 * 10^4`
- Mọi điểm **khác nhau**.

In [1]:
from typing import List, Tuple, Optional
from collections import defaultdict
from itertools import combinations, permutations
import random, time

examples = [
    ([[1, 2], [2, 1], [1, 0], [0, 1]], 2.0),
    ([[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]], 1.0),
    ([[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]], 0.0),
]

def close(a: float, b: float, tol: float = 1e-5) -> bool:
    return abs(a - b) <= tol

for pts, expected in examples:
    print(f"points={pts} kỳ vọng={expected}")

points=[[1, 2], [2, 1], [1, 0], [0, 1]] kỳ vọng=2.0
points=[[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]] kỳ vọng=1.0
points=[[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]] kỳ vọng=0.0


## 3. Kiến thức nền

### 3.1. Tích vô hướng và tích có hướng

Cho vectơ `u = (ux, uy)`, `v = (vx, vy)`:

- tích vô hướng `u · v = ux * vx + uy * vy`; `u ⟂ v` ⇔ `u · v = 0`;
- tích có hướng (độ lớn) `u × v = ux * vy - uy * vx`; diện tích hình bình hành là `|u × v|`.

Với điểm nguyên, cả hai là **số nguyên** nên tính **chính xác**, không cần `float`.

In [2]:
def dot(u, v):
    return u[0] * v[0] + u[1] * v[1]

def cross(u, v):
    return u[0] * v[1] - u[1] * v[0]

u, v = (2, 1), (-1, 2)
print("u · v =", dot(u, v), "(vuông góc)", "| |u × v| =", abs(cross(u, v)), "(diện tích)")

u · v = 0 (vuông góc) | |u × v| = 5 (diện tích)


### 3.2. Hình chữ nhật ⇔ hai đường chéo cắt nhau tại trung điểm và bằng nhau

Tứ giác `A B C D` có hai đường chéo `AC`, `BD`. Nó là hình bình hành ⇔ `AC` và `BD` có **cùng trung điểm** (`A + C = B + D` dạng vectơ). Hình bình hành là hình chữ nhật ⇔ hai đường chéo **bằng nhau** (`|AC|^2 = |BD|^2`). Dùng **tổng tọa độ** `A + C` thay cho trung điểm để tránh số thực.

In [3]:
def is_rectangle(A, B, C, D) -> bool:
    """Bốn điểm A, B, C, D (theo thứ tự đỉnh) có tạo hình chữ nhật không suy biến?"""
    same_center = (A[0] + C[0], A[1] + C[1]) == (B[0] + D[0], B[1] + D[1])
    sq = lambda P, Q: (P[0] - Q[0]) ** 2 + (P[1] - Q[1]) ** 2
    return same_center and sq(A, C) == sq(B, D) and cross((B[0] - A[0], B[1] - A[1]), (D[0] - A[0], D[1] - A[1])) != 0

print(is_rectangle((1, 2), (2, 1), (1, 0), (0, 1)), is_rectangle((0, 0), (2, 0), (3, 1), (1, 1)))

True False


### 3.3. So sánh số thực có sai số

Đề chấp nhận sai số `10^-5`. Với kiểm thử ta so `abs(a - b) <= 1e-5`. Ở đây diện tích là số nguyên nên `float(area)` chính xác với mọi giá trị `<= 2^53` (tối đa `~ 1,6 * 10^9`); dù vậy ta **vẫn so sánh có dung sai** đúng như yêu cầu của đề.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `[[1,2],[2,1],[1,0],[0,1]]`.

- Các đường chéo: `(1,2)-(1,0)` có tổng `(2,2)` và bình phương độ dài `4`; `(2,1)-(0,1)` có tổng `(2,2)` và bình phương độ dài `4`. Cùng khóa `((2, 2), 4)` ⇒ hai đường chéo của một hình chữ nhật.
- Đỉnh `A = (1,2)`, `C = (1,0)`, `B = (2,1)`, `D = (0,1)`: cạnh `AB = (1,-1)`, `AD = (-1,-1)`: tích có hướng `1 * (-1) - (-1) * (-1) = -2`: diện tích `2`.

Ví dụ 3: không có hai đường chéo cùng khóa ⇒ `0`.

## 5. Cách 1 — Brute Force: thử mọi tập 4 điểm theo định nghĩa

**Ý tưởng.** Với mỗi tập 4 điểm `{P0, P1, P2, P3}`, thử các thứ tự đỉnh `P0, Pa, Pb, Pc` (hoán vị của ba điểm còn lại) và kiểm tra **bốn góc vuông** theo định nghĩa: các tích vô hướng của hai cạnh kề tại mỗi đỉnh bằng `0`. Nếu đúng, diện tích là `|AB × AD|` (tích độ dài hai cạnh kề). Lấy nhỏ nhất. Đây là định nghĩa trực tiếp nên cũng là **oracle**, không dựa vào đặc trưng đường chéo.

In [4]:
def min_area_brute(points: List[List[int]]) -> float:
    best = None
    for quad in combinations(points, 4):
        p0 = quad[0]
        for order in permutations(quad[1:]):
            A, B, C, D = p0, order[0], order[1], order[2]
            ring = [A, B, C, D]
            ok = True
            for t in range(4):
                P, Q, R = ring[t - 1], ring[t], ring[(t + 1) % 4]            # góc tại Q
                if dot((P[0] - Q[0], P[1] - Q[1]), (R[0] - Q[0], R[1] - Q[1])) != 0:
                    ok = False
                    break
            if ok:
                area = abs(cross((B[0] - A[0], B[1] - A[1]), (D[0] - A[0], D[1] - A[1])))
                if area > 0 and (best is None or area < best):
                    best = area
    return float(best) if best is not None else 0.0

for pts, expected in examples:
    got = min_area_brute(pts)
    print(pts, "->", got, "OK" if close(got, expected) else "SAI")
    assert close(got, expected)

[[1, 2], [2, 1], [1, 0], [0, 1]] -> 2.0 OK
[[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]] -> 1.0 OK
[[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]] -> 0.0 OK


**Độ phức tạp và nút thắt.** `C(n, 4) * 6` phép kiểm tra: với `n = 50` là `230300 * 6 ~ 1,4 * 10^6` thao tác nhỏ: vẫn chạy được nhưng chậm so với cần thiết, và tăng như `n^4`. Xem thời gian:

In [5]:
rng = random.Random(963)
for n in (10, 20, 30, 40):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(15) for y in range(15)], n)]
    t0 = time.perf_counter()
    min_area_brute(pts)
    print(f"n={n:2}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n=10:       0.8 ms
n=20:      21.7 ms
n=30:      80.7 ms


n=40:     270.8 ms


Nút thắt: ta xét cả những tập 4 điểm hiển nhiên không tạo hình chữ nhật. Nếu chọn **ba** điểm có góc vuông thì đỉnh thứ tư **xác định** (tra bảng băm).

## 6. Ý tưởng thuật toán / bước đột phá

**Hình chữ nhật được xác định bởi một số ít điểm.** Biết ba đỉnh với góc vuông ở đỉnh giữa thì đỉnh thứ tư là `D = A + C - B`: kiểm tra bằng `set` trong `O(1)`: `O(n^3)`. Tốt hơn: **hai đường chéo cùng tâm và cùng độ dài** xác định hình chữ nhật: nhóm mọi cặp điểm theo khóa `(tổng tọa độ, bình phương độ dài)` trong `O(n^2)`, rồi ghép hai cặp cùng nhóm.

| Cách | Ý tưởng |
|---|---|
| 2 | Bộ ba `(A, B, C)` với `BA ⟂ BC`; đỉnh thứ tư `A + C - B` tra trong `set` |
| 3 | Nhóm mọi cặp điểm theo `(tâm, độ dài)` của đường chéo (nộp) |
| 4 | Nhóm các cạnh theo **vectơ**; hai cạnh cùng vectơ tạo hình bình hành; kiểm tra góc vuông |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bộ ba có góc vuông và tra đỉnh thứ tư

**Giải thích từng đoạn.**

- `S = {(x, y)}` để tra `O(1)`.
- Với mỗi đỉnh góc vuông `B` và hai điểm khác `A`, `C` (`A != C`): nếu `(A - B) · (C - B) == 0` và `D = A + C - B` thuộc `S` thì có hình chữ nhật `A B C D` với diện tích `|(A - B) × (C - B)|`.
- Mỗi hình chữ nhật bị đếm nhiều lần (theo đỉnh góc vuông và thứ tự `A`, `C`) nhưng ta chỉ cần `min` nên không sao.
- `A` và `C` khác `B` và khác nhau; nếu `A - B` hoặc `C - B` bằng `0` thì bị loại (điểm khác nhau nên không xảy ra).

In [6]:
def min_area_triples(points: List[List[int]]) -> float:
    pts = [tuple(p) for p in points]
    S = set(pts)
    best = None
    for B in pts:
        for A in pts:
            if A == B:
                continue
            u = (A[0] - B[0], A[1] - B[1])
            for C in pts:
                if C == B or C == A:
                    continue
                v = (C[0] - B[0], C[1] - B[1])
                if u[0] * v[0] + u[1] * v[1] != 0:
                    continue
                D = (A[0] + C[0] - B[0], A[1] + C[1] - B[1])
                if D in S:
                    area = abs(u[0] * v[1] - u[1] * v[0])
                    if best is None or area < best:
                        best = area
    return float(best) if best is not None else 0.0

for pts, expected in examples:
    got = min_area_triples(pts)
    print(pts, "->", got, "OK" if close(got, expected) else "SAI")
    assert close(got, expected)

[[1, 2], [2, 1], [1, 0], [0, 1]] -> 2.0 OK
[[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]] -> 1.0 OK
[[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]] -> 0.0 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        pts = [tuple(p) for p in points]
        S = set(pts)
        best = None
        for B in pts:
            for A in pts:
                if A == B:
                    continue
                ux, uy = A[0] - B[0], A[1] - B[1]
                for C in pts:
                    if C == B or C == A:
                        continue
                    vx, vy = C[0] - B[0], C[1] - B[1]
                    if ux * vx + uy * vy != 0:
                        continue
                    if (A[0] + C[0] - B[0], A[1] + C[1] - B[1]) in S:
                        area = abs(ux * vy - uy * vx)
                        if best is None or area < best:
                            best = area
        return float(best) if best is not None else 0.0

sol_triples = Solution().minAreaFreeRect
assert all(close(sol_triples([list(p) for p in pts]), e) for pts, e in examples)
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n^3)` thời gian (`50^3 = 125000`), `O(n)` bộ nhớ.

---

### 7.2. Cách 3 — Nhóm các đường chéo theo `(tâm, độ dài)` (cách nộp chính)

**Giải thích từng đoạn.**

- Với mọi cặp điểm `(P, Q)` (`P` đứng trước `Q`), tính khóa `(Px + Qx, Py + Qy, (Px - Qx)^2 + (Py - Qy)^2)` toàn số nguyên (tổng tọa độ thay trung điểm, bình phương độ dài thay độ dài).
- Hai cặp **khác nhau** cùng khóa là hai đường chéo của một hình chữ nhật (cùng trung điểm và cùng độ dài, bốn đầu mút phân biệt vì mỗi đường chéo là một cặp khác nhau cùng tâm cùng độ dài).
- Với hai cặp `(A, C)` và `(B, D)` cùng nhóm: diện tích `|(B - A) × (D - A)|`.

In [8]:
def min_area_diagonals(points: List[List[int]]) -> float:
    pts = [tuple(p) for p in points]
    groups = defaultdict(list)
    n = len(pts)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = pts[i], pts[j]
            key = (x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)
            groups[key].append((i, j))
    best = None
    for pairs in groups.values():
        for a in range(len(pairs)):
            for b in range(a + 1, len(pairs)):
                A, C = pts[pairs[a][0]], pts[pairs[a][1]]
                B, D = pts[pairs[b][0]], pts[pairs[b][1]]
                area = abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0]))
                if best is None or area < best:
                    best = area
    return float(best) if best is not None else 0.0

for pts, expected in examples:
    got = min_area_diagonals(pts)
    print(pts, "->", got, "OK" if close(got, expected) else "SAI")
    assert close(got, expected)

[[1, 2], [2, 1], [1, 0], [0, 1]] -> 2.0 OK
[[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]] -> 1.0 OK
[[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]] -> 0.0 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        from collections import defaultdict
        n = len(points)
        groups = defaultdict(list)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
        best = None
        for pairs in groups.values():
            for a in range(len(pairs)):
                ax, ay = points[pairs[a][0]]
                for b in range(a + 1, len(pairs)):
                    bx, by = points[pairs[b][0]]
                    dx, dy = points[pairs[b][1]]
                    area = abs((bx - ax) * (dy - ay) - (by - ay) * (dx - ax))
                    if best is None or area < best:
                        best = area
        return float(best) if best is not None else 0.0

sol_diag = Solution().minAreaFreeRect
assert all(close(sol_diag([list(p) for p in pts]), e) for pts, e in examples)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n^2)` để nhóm; ghép các cặp trong mỗi nhóm tối đa `O(n^2)` mỗi nhóm lớn (tổng thực tế rất nhỏ); bộ nhớ `O(n^2)`.

---

### 7.3. Cách 4 — Nhóm các cạnh theo vectơ

**Ý tưởng khác.** Hình chữ nhật có hai cạnh đối **bằng nhau và song song** (cùng vectơ). Nhóm mọi **cạnh có hướng** `P -> Q` theo vectơ `v = Q - P` (cả hai chiều): `edges[v] = danh sách điểm đầu`. Hai cạnh cùng vectơ `v` với điểm đầu `A` và `D` tạo hình bình hành `A, A + v, D + v, D`; đó là hình chữ nhật ⇔ `(D - A) · v = 0`. Diện tích `|v × (D - A)|`.

**Giải thích từng đoạn.** Mỗi hình chữ nhật xuất hiện nhiều lần (khác thứ tự), ta chỉ cần `min`. Điểm `A` và `D` phải khác nhau, và điều kiện vuông góc loại hình bình hành thường.

In [10]:
def min_area_edges(points: List[List[int]]) -> float:
    pts = [tuple(p) for p in points]
    edges = defaultdict(list)
    for P in pts:
        for Q in pts:
            if P != Q:
                edges[(Q[0] - P[0], Q[1] - P[1])].append(P)
    best = None
    for (vx, vy), starts in edges.items():
        for a in range(len(starts)):
            for b in range(a + 1, len(starts)):
                A, D = starts[a], starts[b]
                w = (D[0] - A[0], D[1] - A[1])
                if vx * w[0] + vy * w[1] == 0:
                    area = abs(vx * w[1] - vy * w[0])
                    if area > 0 and (best is None or area < best):
                        best = area
    return float(best) if best is not None else 0.0

for pts, expected in examples:
    got = min_area_edges(pts)
    print(pts, "->", got, "OK" if close(got, expected) else "SAI")
    assert close(got, expected)

[[1, 2], [2, 1], [1, 0], [0, 1]] -> 2.0 OK
[[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]] -> 1.0 OK
[[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]] -> 0.0 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        from collections import defaultdict
        pts = [tuple(p) for p in points]
        edges = defaultdict(list)
        for P in pts:
            for Q in pts:
                if P != Q:
                    edges[(Q[0] - P[0], Q[1] - P[1])].append(P)
        best = None
        for (vx, vy), starts in edges.items():
            for a in range(len(starts)):
                ax, ay = starts[a]
                for b in range(a + 1, len(starts)):
                    wx, wy = starts[b][0] - ax, starts[b][1] - ay
                    if vx * wx + vy * wy == 0:
                        area = abs(vx * wy - vy * wx)
                        if best is None or area < best:
                            best = area
        return float(best) if best is not None else 0.0

sol_edges = Solution().minAreaFreeRect
assert all(close(sol_edges([list(p) for p in pts]), e) for pts, e in examples)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n^2)` cạnh có hướng; mỗi nhóm cùng vectơ ghép cặp (`O(k^2)`); với `n = 50` tối đa vài triệu thao tác nhỏ; bộ nhớ `O(n^2)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho ví dụ 1: các khóa đường chéo

`points = [[1,2],[2,1],[1,0],[0,1]]` (chỉ số `0..3`). Có 6 cặp:

In [12]:
def trace_diagonals(points):
    groups = defaultdict(list)
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            (x1, y1), (x2, y2) = points[i], points[j]
            key = (x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)
            groups[key].append((i, j))
            print(f"cặp ({i},{j}) = {points[i]}-{points[j]}: tổng tọa độ ({x1 + x2},{y1 + y2}), bình phương độ dài {key[2]}")
    print("các nhóm có >= 2 cặp:", {k: v for k, v in groups.items() if len(v) >= 2})
    return groups

g = trace_diagonals(examples[0][0])
assert any(len(v) == 2 for v in g.values())

cặp (0,1) = [1, 2]-[2, 1]: tổng tọa độ (3,3), bình phương độ dài 2
cặp (0,2) = [1, 2]-[1, 0]: tổng tọa độ (2,2), bình phương độ dài 4
cặp (0,3) = [1, 2]-[0, 1]: tổng tọa độ (1,3), bình phương độ dài 2
cặp (1,2) = [2, 1]-[1, 0]: tổng tọa độ (3,1), bình phương độ dài 2
cặp (1,3) = [2, 1]-[0, 1]: tổng tọa độ (2,2), bình phương độ dài 4
cặp (2,3) = [1, 0]-[0, 1]: tổng tọa độ (1,1), bình phương độ dài 2
các nhóm có >= 2 cặp: {(2, 2, 4): [(0, 2), (1, 3)]}


### 8.2. Cách 3: tính diện tích từ hai đường chéo cùng khóa

Nhóm `((2, 2), 4)` có hai cặp `(0, 2)` (`(1,2)-(1,0)`) và `(1, 3)` (`(2,1)-(0,1)`). `A = (1,2)`, `B = (2,1)`, `D = (0,1)`: `B - A = (1,-1)`, `D - A = (-1,-1)`: `|1 * (-1) - (-1) * (-1)| = |-1 - 1| = 2`.

In [13]:
A, B, D = (1, 2), (2, 1), (0, 1)
area = abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0]))
print("diện tích =", area)
assert area == 2

diện tích = 2


### 8.3. Cách 2 (bộ ba góc vuông) cho ví dụ 2

`B = (1, 1)`, `A = (1, 0)`, `C = (2, 1)`: `A - B = (0, -1)`, `C - B = (1, 0)`: tích vô hướng `0`; `D = A + C - B = (2, 0)` thuộc tập. Diện tích `|0 * 0 - (-1) * 1| = 1`.

In [14]:
S = {tuple(p) for p in examples[1][0]}
B, A, C = (1, 1), (1, 0), (2, 1)
u, v = (A[0] - B[0], A[1] - B[1]), (C[0] - B[0], C[1] - B[1])
D = (A[0] + C[0] - B[0], A[1] + C[1] - B[1])
print("u · v =", dot(u, v), "| D =", D, "thuộc tập:", D in S, "| diện tích:", abs(cross(u, v)))
assert dot(u, v) == 0 and D in S and abs(cross(u, v)) == 1

u · v = 0 | D = (2, 0) thuộc tập: True | diện tích: 1


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (nhóm các đường chéo theo `(tâm, độ dài)`): `O(n^2)`, hoàn toàn bằng số nguyên.

```python
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        from collections import defaultdict
        n = len(points)
        groups = defaultdict(list)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
        best = None
        for pairs in groups.values():
            for a in range(len(pairs)):
                ax, ay = points[pairs[a][0]]
                for b in range(a + 1, len(pairs)):
                    bx, by = points[pairs[b][0]]
                    dx, dy = points[pairs[b][1]]
                    area = abs((bx - ax) * (dy - ay) - (by - ay) * (dx - ax))
                    if best is None or area < best:
                        best = area
        return float(best) if best is not None else 0.0
```

In [15]:
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        from collections import defaultdict
        n = len(points)
        groups = defaultdict(list)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
        best = None
        for pairs in groups.values():
            for a in range(len(pairs)):
                ax, ay = points[pairs[a][0]]
                for b in range(a + 1, len(pairs)):
                    bx, by = points[pairs[b][0]]
                    dx, dy = points[pairs[b][1]]
                    area = abs((bx - ax) * (dy - ay) - (by - ay) * (dx - ax))
                    if best is None or area < best:
                        best = area
        return float(best) if best is not None else 0.0

sol = Solution()
for pts, expected in examples:
    got = sol.minAreaFreeRect([list(p) for p in pts])
    print(pts, "->", got, "OK" if close(got, expected) else "SAI")
    assert close(got, expected)

[[1, 2], [2, 1], [1, 0], [0, 1]] -> 2.0 OK
[[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]] -> 1.0 OK
[[0, 3], [1, 2], [3, 1], [1, 3], [2, 1]] -> 0.0 OK


## 10. Vì sao đúng

**(a) Đặc trưng hình chữ nhật.** Tứ giác `ABCD` (đỉnh theo thứ tự) là hình bình hành ⇔ hai đường chéo `AC`, `BD` có cùng trung điểm ⇔ `A + C = B + D`. Một hình bình hành là hình chữ nhật ⇔ hai đường chéo bằng nhau (`|AC| = |BD|`): thật vậy nếu `A + C = B + D` thì `AC^2 - BD^2 = ... = 4 * (AB · AD)` nên bằng nhau ⇔ `AB ⟂ AD`. Do đó hai cặp điểm **khác nhau** có cùng tổng tọa độ và cùng bình phương độ dài là hai đường chéo của một hình chữ nhật (không suy biến vì bốn đầu mút phân biệt: hai đường chéo phân biệt cùng tâm cùng độ dài không thể trùng hoặc cùng nằm trên một đường thẳng với chung đầu mút). Ngược lại mọi hình chữ nhật tạo hai đường chéo cùng khóa. Cách 3 xét mọi cặp cùng nhóm nên lấy được mọi hình chữ nhật.

**(b) Diện tích.** `Area = |AB| * |AD| = |AB × AD|` (tích có hướng của hai cạnh vuông góc có độ lớn bằng tích độ dài). Với tọa độ nguyên, đây là số nguyên, tính chính xác.

**(c) Cách 2.** Hình chữ nhật `ABCD` với góc vuông tại `B`: `D = A + C - B` (hình bình hành) nên tra `D` trong `set` là đúng; mỗi hình chữ nhật được tìm thấy khi chọn một đỉnh làm `B` và hai đỉnh kề làm `A`, `C`.

**(d) Cách 4.** Hai cạnh cùng vectơ `v` có điểm đầu `A != D` tạo hình bình hành `A, A + v, D + v, D`; nó là hình chữ nhật ⇔ `v ⟂ (D - A)` (góc tại `A` vuông). Mọi hình chữ nhật có hai cạnh đối cùng vectơ nên được liệt kê.

**(e) Sai số.** Kết quả đúng là số nguyên; trả về `float(area)` là chính xác; đề cho phép sai số `10^-5` nên so sánh trong kiểm thử dùng dung sai.

**Kiểm chứng bằng máy.** Oracle **độc lập** với đặc trưng đường chéo: Cách 1 kiểm tra **bốn góc vuông** theo định nghĩa trên mọi tập 4 điểm. So mọi cách (có dung sai `1e-5`) trên điểm ngẫu nhiên (lưới nhỏ, nhiều hình chữ nhật), điểm có hình chữ nhật nghiêng được cài sẵn (tọa độ lớn tới `4 * 10^4`), và với `n = 50`. Thêm một **oracle chính xác bằng phân số** (`fractions.Fraction`) cho diện tích để chắc chắn không có sai số số thực.

In [16]:
from fractions import Fraction

def min_area_exact(points: List[List[int]]) -> Fraction:
    """Oracle chính xác: dùng Fraction và định nghĩa bốn góc vuông qua tích vô hướng (độc lập với Cách 3)."""
    best = None
    for quad in combinations(points, 4):
        p0 = quad[0]
        for order in permutations(quad[1:]):
            ring = [p0, order[0], order[1], order[2]]
            if all(dot((ring[t - 1][0] - ring[t][0], ring[t - 1][1] - ring[t][1]),
                       (ring[(t + 1) % 4][0] - ring[t][0], ring[(t + 1) % 4][1] - ring[t][1])) == 0 for t in range(4)):
                A, B, D = ring[0], ring[1], ring[3]
                area = Fraction(abs(cross((B[0] - A[0], B[1] - A[1]), (D[0] - A[0], D[1] - A[1]))))
                if area > 0 and (best is None or area < best):
                    best = area
    return best if best is not None else Fraction(0)

rng = random.Random(1963)
for _ in range(200):
    n = rng.randint(1, 12)
    pts = [list(p) for p in rng.sample([(x, y) for x in range(6) for y in range(6)], n)]
    ex = min_area_exact(pts)
    for fn in (min_area_brute, min_area_triples, min_area_diagonals, min_area_edges):
        assert close(fn(pts), float(ex)), (pts, ex)
print("oracle Fraction và bốn cách khớp (có dung sai) trên 200 tập điểm ngẫu nhiên")

oracle Fraction và bốn cách khớp (có dung sai) trên 200 tập điểm ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập 4 điểm | `O(n^4)` | `O(1)` |
| 2. Bộ ba góc vuông | `O(n^3)` | `O(n)` |
| 3. Nhóm đường chéo | `O(n^2)` (+ ghép cặp trong nhóm) | `O(n^2)` |
| 4. Nhóm cạnh theo vectơ | `O(n^2)` (+ ghép cặp trong nhóm) | `O(n^2)` |

Đo thử với `n = 50` (cỡ tối đa), điểm trên lưới nhỏ (nhiều hình chữ nhật) và điểm ngẫu nhiên thưa:

In [17]:
rng = random.Random(11)
dense = [list(p) for p in rng.sample([(x, y) for x in range(8) for y in range(8)], 50)]
sparse = [[rng.randint(0, 40000), rng.randint(0, 40000)] for _ in range(50)]
for label, pts in (("lưới dày 8x8 (50 điểm)", dense), ("thưa tọa độ tới 4*10^4", sparse)):
    ref = min_area_diagonals(pts)
    for name, fn in (("Cách 2", sol_triples), ("Cách 3", sol_diag), ("Cách 4", sol_edges), ("nộp", Solution().minAreaFreeRect)):
        t0 = time.perf_counter()
        got = fn([list(p) for p in pts])
        dt = (time.perf_counter() - t0) * 1000
        assert close(got, ref), name
        print(f"{label:28} {name:7}: {dt:8.2f} ms (đáp án {got})")
t0 = time.perf_counter()
b = min_area_brute(dense)
print(f"{'lưới dày':28} Cách 1  : {1000 * (time.perf_counter() - t0):8.1f} ms (đáp án {b})")
assert close(b, min_area_diagonals(dense))

lưới dày 8x8 (50 điểm)       Cách 2 :    14.71 ms (đáp án 1.0)
lưới dày 8x8 (50 điểm)       Cách 3 :     0.62 ms (đáp án 1.0)
lưới dày 8x8 (50 điểm)       Cách 4 :     2.24 ms (đáp án 1.0)
lưới dày 8x8 (50 điểm)       nộp    :     0.54 ms (đáp án 1.0)


thưa tọa độ tới 4*10^4       Cách 2 :    16.83 ms (đáp án 0.0)
thưa tọa độ tới 4*10^4       Cách 3 :     0.73 ms (đáp án 0.0)
thưa tọa độ tới 4*10^4       Cách 4 :     1.19 ms (đáp án 0.0)
thưa tọa độ tới 4*10^4       nộp    :     0.62 ms (đáp án 0.0)


lưới dày                     Cách 1  :    712.8 ms (đáp án 1.0)


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle (bốn góc vuông) **bằng so sánh có dung sai**, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [18]:
def solutions_list():
    return {
        "triples": min_area_triples,
        "diagonals": min_area_diagonals,
        "edges": min_area_edges,
        "sol_triples": sol_triples,
        "sol_diag": sol_diag,
        "sol_edges": sol_edges,
        "final": Solution().minAreaFreeRect,
    }

def run_all(points, expected=None, brute_limit=24):
    pts = [list(p) for p in points]
    ref = float(min_area_exact(pts)) if len(pts) <= brute_limit else min_area_diagonals(pts)
    for name, fn in solutions_list().items():
        got = fn([list(p) for p in pts])
        assert close(got, ref), (name, pts, got, ref)
    if expected is not None:
        assert close(ref, expected), ("kỳ vọng sai", pts, expected, ref)
    return ref

# 12.1 ví dụ đề
for pts, expected in examples:
    assert close(run_all(pts, expected), expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[0, 0]], 0.0),                                         # một điểm
    ([[0, 0], [1, 1]], 0.0),                                 # hai điểm
    ([[0, 0], [1, 0], [0, 1]], 0.0),                         # ba điểm
    ([[0, 0], [1, 0], [0, 1], [1, 1]], 1.0),                 # hình vuông đơn vị
    ([[0, 0], [2, 0], [0, 1], [2, 1]], 2.0),                 # chữ nhật 2 x 1
    ([[0, 0], [1, 0], [2, 0], [3, 0]], 0.0),                 # thẳng hàng
    ([[1, 0], [3, 1], [2, 3], [0, 2]], 5.0),                 # hình vuông nghiêng cạnh (2, 1): diện tích 5
    ([[0, 0], [2, 1], [3, 3], [1, 2]], None),                # hình bình hành không phải chữ nhật?
    ([[0, 0], [4, 0], [4, 3], [0, 3], [2, 1]], 12.0),        # chữ nhật 4 x 3 và một điểm thừa
    ([[0, 0], [3, 4], [-0 + 4, 3], [7, 7]], None),           # nghiêng (3,4)-(4,-3)? kiểm bằng oracle
    ([[0, 1], [1, 0], [2, 1], [1, 2], [1, 1]], 2.0),         # hình vuông nghiêng đường chéo 2 (diện tích 2), tâm là điểm thừa
    ([[0, 0], [40000, 0], [0, 40000], [40000, 40000]], 1600000000.0),   # biên: 4*10^4 x 4*10^4
    ([[0, 0], [1, 2], [3, 1], [2, 3], [4, 2]], None),
]
for pts, expected in edge:
    run_all(pts, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_floor_midpoint(points):                 # trung điểm bằng // (mất thông tin khi tổng lẻ)
    n = len(points)
    groups = defaultdict(list)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = points[i], points[j]
            groups[((x1 + x2) // 2, (y1 + y2) // 2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
    best = None
    for pairs in groups.values():
        for a in range(len(pairs)):
            for b in range(a + 1, len(pairs)):
                A, C = points[pairs[a][0]], points[pairs[a][1]]
                B, D = points[pairs[b][0]], points[pairs[b][1]]
                area = abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0]))
                if area and (best is None or area < best):
                    best = area
    return float(best) if best is not None else 0.0

def wrong_parallelogram(points):                  # chỉ nhóm theo tâm (bỏ độ dài): nhận hình bình hành
    n = len(points)
    groups = defaultdict(list)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = points[i], points[j]
            groups[(x1 + x2, y1 + y2)].append((i, j))
    best = None
    for pairs in groups.values():
        for a in range(len(pairs)):
            for b in range(a + 1, len(pairs)):
                A, C = points[pairs[a][0]], points[pairs[a][1]]
                B, D = points[pairs[b][0]], points[pairs[b][1]]
                area = abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0]))
                if area and (best is None or area < best):
                    best = area
    return float(best) if best is not None else 0.0

def wrong_axis_aligned(points):                   # chỉ xét hình chữ nhật song song trục (bài 939)
    S = {tuple(p) for p in points}
    best = None
    for (x1, y1), (x2, y2) in combinations(S, 2):
        if x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S:
            area = abs(x1 - x2) * abs(y1 - y2)
            if best is None or area < best:
                best = area
    return float(best) if best is not None else 0.0

def wrong_inf(points):                            # khởi tạo inf và quên đổi về 0
    r = min_area_diagonals(points)
    return float("inf") if r == 0.0 else r

def find_counter(wrong, tries=20000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(4, 9)
        pts = [list(p) for p in rng.sample([(x, y) for x in range(5) for y in range(5)], n)]
        if not close(wrong([list(p) for p in pts]), float(min_area_exact(pts))):
            return pts
    return None

counters = {}
for name, fn in (("trung điểm bằng //", wrong_floor_midpoint), ("nhận hình bình hành (bỏ độ dài)", wrong_parallelogram),
                 ("chỉ hình chữ nhật song song trục", wrong_axis_aligned), ("trả inf khi không có", wrong_inf)):
    pts = find_counter(fn)
    assert pts is not None, name
    counters[name] = pts
    print(f"{name:34} pts={pts}: sai -> {fn([list(p) for p in pts])}, đúng -> {min_area_exact(pts)}")
    run_all(pts)
print("12.3 phản ví dụ: OK")

trung điểm bằng //                 pts=[[0, 2], [4, 2], [2, 2], [0, 3], [1, 4], [1, 1], [1, 2], [2, 3]]: sai -> 1.0, đúng -> 2
nhận hình bình hành (bỏ độ dài)    pts=[[3, 4], [2, 1], [1, 3], [0, 4], [1, 0], [0, 0], [2, 0]]: sai -> 5.0, đúng -> 0
chỉ hình chữ nhật song song trục   pts=[[1, 0], [3, 4], [4, 3], [0, 1], [1, 3]]: sai -> 0.0, đúng -> 6
trả inf khi không có               pts=[[3, 4], [2, 1], [1, 3], [0, 4], [1, 0], [0, 0], [2, 0]]: sai -> inf, đúng -> 0
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ: lưới 5x5 / 6x6, n tới 16, so với oracle Fraction (bốn góc vuông)
rng = random.Random(4963)
nonzero = 0
for _ in range(400):
    side = rng.choice([4, 5, 6])
    n = rng.randint(1, 14)
    pts = [list(p) for p in rng.sample([(x, y) for x in range(side) for y in range(side)], min(n, side * side))]
    nonzero += run_all(pts) > 0
print(f"12.4 ngẫu nhiên lưới nhỏ (400 ca): OK, {nonzero} ca có hình chữ nhật")

# 12.5 hình chữ nhật nghiêng cài sẵn, toạ độ tới 4*10^4, thêm nhiễu: đáp án <= diện tích hình cài
def planted(rng, extra):
    while True:
        a, b = rng.randint(1, 150), rng.randint(0, 150)
        k = rng.randint(1, 8)
        v = (a, b)
        w = (-b * k, a * k)
        origin = (rng.randint(0, 40000), rng.randint(0, 40000))
        corners = [origin, (origin[0] + v[0], origin[1] + v[1]), (origin[0] + v[0] + w[0], origin[1] + v[1] + w[1]), (origin[0] + w[0], origin[1] + w[1])]
        if len(set(corners)) == 4 and all(0 <= x <= 40000 and 0 <= y <= 40000 for x, y in corners):
            area = abs(v[0] * w[1] - v[1] * w[0])
            pts = set(corners)
            while len(pts) < 4 + extra:
                pts.add((rng.randint(0, 40000), rng.randint(0, 40000)))
            pts = [list(p) for p in pts]
            rng.shuffle(pts)
            return pts, area

rng = random.Random(5963)
for _ in range(150):
    pts, area = planted(rng, rng.randint(0, 20))
    ref = run_all(pts)
    assert ref > 0 and ref <= area + 1e-9, (pts, ref, area)
print("12.5 hình chữ nhật nghiêng cài sẵn (150 ca, toạ độ tới 4*10^4): OK")

# 12.6 n = 50: điểm ngẫu nhiên thưa (đáp án thường 0), lưới dày và cài sẵn; so các cách với nhau
rng = random.Random(6963)
for _ in range(8):
    pts = [[rng.randint(0, 40000), rng.randint(0, 40000)] for _ in range(50)]
    pts = [list(p) for p in {tuple(p) for p in pts}]
    run_all(pts)
for _ in range(6):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(9) for y in range(9)], 50)]
    run_all(pts, brute_limit=0)
print("12.6 n = 50: OK")

# 12.7 tính chất: tịnh tiến, đảo trục, hoán vị thứ tự điểm không đổi đáp án; thêm điểm không làm đáp án tăng
rng = random.Random(7963)
for _ in range(200):
    n = rng.randint(4, 12)
    pts = [list(p) for p in rng.sample([(x, y) for x in range(6) for y in range(6)], n)]
    base = Solution().minAreaFreeRect([list(p) for p in pts])
    shifted = [[x + 7, y + 3] for x, y in pts]
    swapped = [[y, x] for x, y in pts]
    shuffled = pts[:]
    rng.shuffle(shuffled)
    assert close(Solution().minAreaFreeRect(shifted), base) and close(Solution().minAreaFreeRect(swapped), base)
    assert close(Solution().minAreaFreeRect(shuffled), base)
    extra = [list(p) for p in {(x, y) for x in range(7) for y in range(7)} - {tuple(p) for p in pts}]
    if extra:
        more = Solution().minAreaFreeRect(pts + [rng.choice(extra)])
        assert base == 0 or more <= base + 1e-9 and more > 0
print("12.7 tính chất (tịnh tiến, đổi trục, hoán vị, thêm điểm): OK")

12.4 ngẫu nhiên lưới nhỏ (400 ca): OK, 181 ca có hình chữ nhật


12.5 hình chữ nhật nghiêng cài sẵn (150 ca, toạ độ tới 4*10^4): OK


12.6 n = 50: OK
12.7 tính chất (tịnh tiến, đổi trục, hoán vị, thêm điểm): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trung điểm bằng `//` (chia nguyên)

`(x1 + x2) // 2` mất phần lẻ: hai đường chéo có tổng khác nhau (một chẵn, một lẻ) có thể bị coi là cùng tâm.

In [21]:
pts = counters["trung điểm bằng //"]
print("pts =", pts, "| sai ->", wrong_floor_midpoint([list(p) for p in pts]), "| đúng ->", Solution().minAreaFreeRect([list(p) for p in pts]))
assert not close(wrong_floor_midpoint([list(p) for p in pts]), Solution().minAreaFreeRect([list(p) for p in pts]))

pts = [[0, 2], [4, 2], [2, 2], [0, 3], [1, 4], [1, 1], [1, 2], [2, 3]] | sai -> 1.0 | đúng -> 2.0


Vì sao sai: `(1 + 2) // 2 = 1` và `(1 + 1) // 2 = 1` trùng nhau dù trung điểm khác nhau (`1,5` và `1`). **Đúng:** dùng **tổng tọa độ** (`x1 + x2`) làm khóa, không chia.

### Lỗi 2 — Chỉ nhóm theo tâm (bỏ qua độ dài)

Hai đường chéo cùng trung điểm tạo **hình bình hành**; muốn hình chữ nhật cần thêm điều kiện **cùng độ dài**.

In [22]:
pts = counters["nhận hình bình hành (bỏ độ dài)"]
print("pts =", pts, "| sai ->", wrong_parallelogram([list(p) for p in pts]), "| đúng ->", Solution().minAreaFreeRect([list(p) for p in pts]))
assert not close(wrong_parallelogram([list(p) for p in pts]), Solution().minAreaFreeRect([list(p) for p in pts]))

pts = [[3, 4], [2, 1], [1, 3], [0, 4], [1, 0], [0, 0], [2, 0]] | sai -> 5.0 | đúng -> 0.0


Vì sao sai: nhận cả hình bình hành thường. **Đúng:** khóa gồm `(tổng x, tổng y, bình phương độ dài)`.

### Lỗi 3 — Chỉ xét hình chữ nhật song song trục (bài 939)

Bài **939** (Minimum Area Rectangle) giới hạn cạnh song song trục; bài này (**963**) cho phép **mọi hướng**.

In [23]:
pts = counters["chỉ hình chữ nhật song song trục"]
print("pts =", pts, "| sai ->", wrong_axis_aligned([list(p) for p in pts]), "| đúng ->", Solution().minAreaFreeRect([list(p) for p in pts]))
assert not close(wrong_axis_aligned([list(p) for p in pts]), Solution().minAreaFreeRect([list(p) for p in pts]))
print("ví dụ 1 (hình vuông nghiêng):", wrong_axis_aligned(examples[0][0]), "(sai) so với", Solution().minAreaFreeRect([list(p) for p in examples[0][0]]))

pts = [[1, 0], [3, 4], [4, 3], [0, 1], [1, 3]] | sai -> 0.0 | đúng -> 6.0
ví dụ 1 (hình vuông nghiêng): 0.0 (sai) so với 2.0


Vì sao sai: bỏ sót các hình chữ nhật nghiêng như ví dụ 1. **Đúng:** dùng đặc trưng đường chéo (hoặc góc vuông) cho hướng bất kỳ.

### Lỗi 4 — Trả `inf` khi không có hình chữ nhật

Khởi tạo `best = float("inf")` và trả về nguyên giá trị đó. Đề yêu cầu `0`.

In [24]:
pts = counters["trả inf khi không có"]
print("pts =", pts, "| sai ->", wrong_inf([list(p) for p in pts]), "| đúng ->", Solution().minAreaFreeRect([list(p) for p in pts]))
assert wrong_inf([list(p) for p in pts]) == float("inf") and Solution().minAreaFreeRect([list(p) for p in pts]) == 0.0

pts = [[3, 4], [2, 1], [1, 3], [0, 4], [1, 0], [0, 0], [2, 0]] | sai -> inf | đúng -> 0.0


Vì sao sai: ví dụ 3 có đáp án `0`. **Đúng:** `return 0.0 if best is None else float(best)`.

### Lỗi 5 — Dùng số thực để tính độ dài và so sánh bằng `==`

`math.sqrt` rồi so sánh `==` dễ sai do làm tròn. **Đúng:** so sánh **bình phương độ dài** (số nguyên).

In [25]:
import math
a, b = 2, 8
print("sqrt(2) * sqrt(8) == 4 ?", math.sqrt(a) * math.sqrt(b) == 4.0, "| bình phương nguyên: 2 * 8 == 16 ?", a * b == 16)

sqrt(2) * sqrt(8) == 4 ? False | bình phương nguyên: 2 * 8 == 16 ? True


Vì sao: tích hai căn có thể lệch một ulp. **Đúng:** làm việc với bình phương nguyên và tích có hướng nguyên.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "hình chữ nhật/vuông từ tập điểm, hướng bất kỳ" ⇒ **đường chéo cùng tâm, cùng độ dài** (nhóm bằng bảng băm) hoặc **bộ ba có góc vuông**.
- **Toàn số nguyên:** dùng `x1 + x2` (không chia 2), bình phương độ dài, tích vô hướng và tích có hướng; chỉ đổi sang `float` khi trả.
- **Diện tích** = `|u × v|` với `u`, `v` là hai cạnh kề.
- Khi đáp án là số thực: so sánh **có dung sai** (`abs(a - b) <= 1e-5`) và kiểm tra thêm bằng `Fraction`.
- Cùng họ: Minimum Area Rectangle (939), Valid Square (593), Max Points on a Line (149), Number of Boomerangs (447).
- Test nên có: hình vuông nghiêng, chữ nhật song song trục, hình bình hành (không chữ nhật), điểm thẳng hàng, tọa độ `4 * 10^4`.

## 15. Biến thể

1. **Diện tích lớn nhất** thay vì nhỏ nhất: thay `min` bằng `max`.
2. **Đếm số hình chữ nhật** (bài tập 5): mỗi hình chữ nhật có đúng hai đường chéo cùng nhóm.
3. **Song song trục (939):** `O(n^2)` bằng bảng băm theo cột.
4. Bài liên quan: 939, 593.

Dưới đây là biến thể 1 (diện tích lớn nhất) kiểm bằng oracle Fraction.

In [26]:
def max_area_diagonals(points: List[List[int]]) -> float:
    pts = [tuple(p) for p in points]
    groups = defaultdict(list)
    n = len(pts)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = pts[i], pts[j]
            groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
    best = 0
    for pairs in groups.values():
        for a in range(len(pairs)):
            for b in range(a + 1, len(pairs)):
                A, B, D = pts[pairs[a][0]], pts[pairs[b][0]], pts[pairs[b][1]]
                best = max(best, abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0])))
    return float(best)

def max_area_exact(points):
    best = 0
    for quad in combinations(points, 4):
        p0 = quad[0]
        for order in permutations(quad[1:]):
            ring = [p0, order[0], order[1], order[2]]
            if all(dot((ring[t - 1][0] - ring[t][0], ring[t - 1][1] - ring[t][1]),
                       (ring[(t + 1) % 4][0] - ring[t][0], ring[(t + 1) % 4][1] - ring[t][1])) == 0 for t in range(4)):
                A, B, D = ring[0], ring[1], ring[3]
                best = max(best, abs(cross((B[0] - A[0], B[1] - A[1]), (D[0] - A[0], D[1] - A[1]))))
    return best

rng = random.Random(15)
for _ in range(150):
    n = rng.randint(4, 12)
    pts = [list(p) for p in rng.sample([(x, y) for x in range(6) for y in range(6)], n)]
    assert close(max_area_diagonals(pts), float(max_area_exact(pts))), pts
print("diện tích lớn nhất của ví dụ 2:", max_area_diagonals(examples[1][0]))

diện tích lớn nhất của ví dụ 2: 1.0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập 4 điểm | Bốn góc vuông | `O(n^4)` | `O(1)` | Oracle |
| 2. Bộ ba góc vuông | `D = A + C - B` | `O(n^3)` | `O(n)` | Đơn giản, đủ cho `n <= 50` |
| 3. Nhóm đường chéo | `(tâm, độ dài)` | `O(n^2)` | `O(n^2)` | **Cách nộp chính** |
| 4. Nhóm cạnh theo vectơ | Hình bình hành + vuông | `O(n^2)` | `O(n^2)` | Cách nhìn khác |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(dot((2, 1), (-1, 2)), cross((2, 1), (-1, 2)))
print(is_rectangle((0, 0), (2, 0), (2, 1), (0, 1)))
print(close(1 / 3 * 3, 1.0), close(0.1 + 0.2, 0.3))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `points = [[1,0],[3,1],[2,3],[0,2]]`: ghi diện tích hình chữ nhật (nếu có).

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành nhóm các đường chéo theo `(tâm, độ dài)`.

In [29]:
def exercise_3_diagonals(points: List[List[int]]) -> float:
    n = len(points)
    groups = defaultdict(list)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = points[i], points[j]
            # TODO 1: khóa = (x1 + x2, y1 + y2, bình phương độ dài); thêm (i, j) vào groups[khóa]
            pass
    best = None
    for pairs in groups.values():
        # TODO 2: với mỗi hai cặp khác nhau trong nhóm tính diện tích |(B - A) x (D - A)| và cập nhật best
        pass
    return float(best) if best is not None else 0.0


# for pts, expected in examples:
#     assert close(exercise_3_diagonals(pts), expected)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_triples(points)` (bộ ba có góc vuông và tra đỉnh thứ tư).

In [30]:
def exercise_4_triples(points: List[List[int]]) -> float:
    # TODO
    pass


# for pts, expected in examples:
#     assert close(exercise_4_triples(pts), expected)

### Bài 5 — Mini challenge: đếm số hình chữ nhật (khó)

Viết `exercise_5_count(points)` trả về **số hình chữ nhật** (không suy biến, tập 4 đỉnh phân biệt) có 4 đỉnh trong tập điểm (hướng bất kỳ).

```python
exercise_5_count([[0,0],[1,0],[0,1],[1,1]])                    -> 1
exercise_5_count([[0,1],[2,1],[1,1],[1,0],[2,0]])              -> 1
exercise_5_count([[0,0],[1,0],[2,0],[0,1],[1,1],[2,1]])        -> 3     # 2 hình 1x1 và 1 hình 2x1
```
Gợi ý: mỗi hình chữ nhật có đúng **hai** đường chéo cùng nhóm: đếm cặp trong từng nhóm.

In [31]:
def exercise_5_count(points: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_count([[0, 0], [1, 0], [0, 1], [1, 1]]) == 1
# assert exercise_5_count([[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]]) == 1
# assert exercise_5_count([[0, 0], [1, 0], [2, 0], [0, 1], [1, 1], [2, 1]]) == 3

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `(2,1) · (-1,2) = 2 * (-1) + 1 * 2`; tích có hướng `2 * 2 - 1 * (-1)`.
- Mức 2: `is_rectangle` kiểm tra cùng tâm và cùng bình phương độ dài đường chéo.
- Mức 3: `0 5`, `True`, `True False` (`0.1 + 0.2` khác `0.3` hơn `1e-5`? thực tế sai khác `~ 5e-17` nên cũng `True`: kiểm tra bằng máy).

**Bài 2**
- Mức 1: cạnh `AB = (2, 1)`, `AD = (-1, 2)`.
- Mức 2: tích vô hướng `0`: góc vuông; tích có hướng `2 * 2 - 1 * (-1) = 5`.
- Mức 3: `5.0`.

**Bài 3**
- Mức 1: `groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))`.
- Mức 2: hai vòng lặp `a < b` trong từng nhóm.
- Mức 3: `area = abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0]))`.

**Bài 4**
- Mức 1: ba vòng lặp `B`, `A`, `C`.
- Mức 2: kiểm tra `(A - B) · (C - B) == 0`.
- Mức 3: `D = (A[0] + C[0] - B[0], A[1] + C[1] - B[1])` thuộc `set`.

**Bài 5**
- Mức 1: nhóm đường chéo như Cách 3.
- Mức 2: một nhóm có `k` cặp cho `C(k, 2)` hình chữ nhật.
- Mức 3: cộng `k * (k - 1) // 2` trên mọi nhóm.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(dot((2, 1), (-1, 2)), cross((2, 1), (-1, 2)))
print(is_rectangle((0, 0), (2, 0), (2, 1), (0, 1)))
print(close(1 / 3 * 3, 1.0), close(0.1 + 0.2, 0.3))

assert (dot((2, 1), (-1, 2)), cross((2, 1), (-1, 2))) == (0, 5)
assert is_rectangle((0, 0), (2, 0), (2, 1), (0, 1)) is True
assert close(1 / 3 * 3, 1.0) and close(0.1 + 0.2, 0.3)        # sai số rất nhỏ hơn dung sai 1e-5

0 5
True
True True


### Đáp án Bài 2

`AB = (2, 1)`, `AD = (-1, 2)`: vuông góc; diện tích `|2 * 2 - 1 * (-1)| = 5`.

In [33]:
pts = [[1, 0], [3, 1], [2, 3], [0, 2]]
hand_answer_solution = 5.0
assert close(Solution().minAreaFreeRect([list(p) for p in pts]), hand_answer_solution)
assert close(float(min_area_exact(pts)), hand_answer_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_diagonals_solution(points: List[List[int]]) -> float:
    n = len(points)
    groups = defaultdict(list)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = points[i], points[j]
            groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
    best = None
    for pairs in groups.values():
        for a in range(len(pairs)):
            for b in range(a + 1, len(pairs)):
                A = points[pairs[a][0]]
                B, D = points[pairs[b][0]], points[pairs[b][1]]
                area = abs((B[0] - A[0]) * (D[1] - A[1]) - (B[1] - A[1]) * (D[0] - A[0]))
                if best is None or area < best:
                    best = area
    return float(best) if best is not None else 0.0

for pts, expected in examples:
    assert close(exercise_3_diagonals_solution(pts), expected)
rng = random.Random(3)
for _ in range(200):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(6) for y in range(6)], rng.randint(1, 12))]
    assert close(exercise_3_diagonals_solution(pts), float(min_area_exact(pts)))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_triples_solution(points: List[List[int]]) -> float:
    pts = [tuple(p) for p in points]
    S = set(pts)
    best = None
    for B in pts:
        for A in pts:
            if A == B:
                continue
            for C in pts:
                if C == B or C == A:
                    continue
                if (A[0] - B[0]) * (C[0] - B[0]) + (A[1] - B[1]) * (C[1] - B[1]) != 0:
                    continue
                if (A[0] + C[0] - B[0], A[1] + C[1] - B[1]) in S:
                    area = abs((A[0] - B[0]) * (C[1] - B[1]) - (A[1] - B[1]) * (C[0] - B[0]))
                    if best is None or area < best:
                        best = area
    return float(best) if best is not None else 0.0

for pts, expected in examples:
    assert close(exercise_4_triples_solution(pts), expected)
rng = random.Random(4)
for _ in range(200):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(6) for y in range(6)], rng.randint(1, 12))]
    assert close(exercise_4_triples_solution(pts), float(min_area_exact(pts)))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_count_solution(points: List[List[int]]) -> int:
    n = len(points)
    groups = defaultdict(int)
    for i in range(n):
        for j in range(i + 1, n):
            (x1, y1), (x2, y2) = points[i], points[j]
            groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)] += 1
    return sum(k * (k - 1) // 2 for k in groups.values())

def count_rectangles_brute(points):
    seen = set()
    for quad in combinations(points, 4):
        p0 = quad[0]
        for order in permutations(quad[1:]):
            ring = [p0, order[0], order[1], order[2]]
            if all(dot((ring[t - 1][0] - ring[t][0], ring[t - 1][1] - ring[t][1]),
                       (ring[(t + 1) % 4][0] - ring[t][0], ring[(t + 1) % 4][1] - ring[t][1])) == 0 for t in range(4)):
                A, B, D = ring[0], ring[1], ring[3]
                if cross((B[0] - A[0], B[1] - A[1]), (D[0] - A[0], D[1] - A[1])) != 0:
                    seen.add(frozenset(map(tuple, quad)))
    return len(seen)

assert exercise_5_count_solution([[0, 0], [1, 0], [0, 1], [1, 1]]) == 1
assert exercise_5_count_solution([[0, 1], [2, 1], [1, 1], [1, 0], [2, 0]]) == 1
assert exercise_5_count_solution([[0, 0], [1, 0], [2, 0], [0, 1], [1, 1], [2, 1]]) == 3
rng = random.Random(5)
for _ in range(200):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(5) for y in range(5)], rng.randint(1, 12))]
    assert exercise_5_count_solution(pts) == count_rectangles_brute(pts), pts
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (nhóm các đường chéo)

```python
class Solution:
    def minAreaFreeRect(self, points: List[List[int]]) -> float:
        from collections import defaultdict
        n = len(points)
        groups = defaultdict(list)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                # khóa toàn số nguyên: (tổng tọa độ = 2 * tâm, bình phương độ dài)
                groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)].append((i, j))
        best = None
        for pairs in groups.values():
            for a in range(len(pairs)):
                ax, ay = points[pairs[a][0]]
                for b in range(a + 1, len(pairs)):
                    bx, by = points[pairs[b][0]]
                    dx, dy = points[pairs[b][1]]
                    area = abs((bx - ax) * (dy - ay) - (by - ay) * (dx - ax))   # |AB x AD|
                    if best is None or area < best:
                        best = area
        return float(best) if best is not None else 0.0                          # không có -> 0
```

Công thức nhớ nhanh: hình chữ nhật ⇔ hai đường chéo cùng tâm và cùng độ dài; diện tích `|AB × AD|` (số nguyên).

### Checklist trước khi submit

- [ ] Khóa là **tổng tọa độ** (không chia 2) và **bình phương** độ dài (không dùng căn)?
- [ ] Khóa gồm cả độ dài (không chỉ tâm, tránh hình bình hành)?
- [ ] Diện tích bằng tích có hướng (số nguyên), đổi `float` khi trả?
- [ ] Không có hình chữ nhật thì trả `0` (không `inf`)?
- [ ] Hình chữ nhật nghiêng được xét (không chỉ song song trục)?
- [ ] Đã thử: ví dụ 1-3, hình bình hành, điểm thẳng hàng, tọa độ `4 * 10^4`, `n = 50`?
- [ ] Tên method đúng `minAreaFreeRect(self, points)` và trả `float`?